# Sentence & Document Embeddings: build it, measure it, see the geometry

One vector for a whole passage, arranged so that **semantic similarity = vector proximity**. This
notebook walks the chapter's claims in runnable order and **measures each one** with the same code
the page and the figures use — every function is imported from `sentence_embeddings.py`, so nothing
here can drift from the prose.

We will, in order:

1. feel **why mean-pooling word vectors fails** — it is *order-blind* (`"dog bit man"` = `"man bit dog"`);
2. fix frequency-domination with **SIF** weighting on a toy example;
3. watch a trained **bi-encoder (Sentence-BERT)** separate paraphrases from unrelated text by cosine;
4. compare **pooling strategies** (mean vs max vs CLS) by STS Spearman;
5. run a small **semantic search**, contrast a **bi-encoder vs a cross-encoder**, and an **STS evaluation**.

> **Backend & determinism.** We load `all-MiniLM-L6-v2` when it is reachable (REAL measured numbers,
> the exact ones on the page). If it is not (offline / no package), the code transparently falls back
> to a small, deterministic **synthetic** encoder so this notebook **always runs**. Execution is pinned
> to **CPU** and seeded, so the numbers are reproducible. Which backend ran is printed honestly.

## Step 0 — load the backend and report the device honestly

`load_encoder()` returns a real Sentence-BERT if available, else the synthetic fallback. The device
line states the detected accelerator and the pinned execution device (CPU, for reproducibility).

In [1]:
import numpy as np

import sentence_embeddings as se

print(se.device_report())
print("torch:", se.torch.__version__, " numpy:", np.__version__)

encoder = se.load_encoder()
backend = "real (all-MiniLM-L6-v2)" if encoder.is_real else "synthetic fallback"
print("backend:", backend)

device: cpu (detected mps; pinned to CPU for reproducibility)
torch: 2.12.0  numpy: 2.4.6


backend: real (all-MiniLM-L6-v2)


## Step 1 — the failure that motivates everything: mean-pooling is order-blind

The most obvious sentence vector is the **average of its word vectors**. It is a strong *topic*
baseline, but averaging is **permutation-invariant**: a bag of the same words gives the same vector.
So `"The dog bit the man."` and `"The man bit the dog."` — same words, opposite meaning — collapse to
the **identical** vector (cosine exactly `1.000`). A real encoder, which *reads the sentence in order*,
must do better.

We assert the contract first (mean-pool is identical; the encoder separates), then print the numbers.

In [2]:
ob = se.order_blindness_demo(encoder)

# Contract: mean-pool cannot tell the two apart (cosine ~ 1.000); the order-aware encoder can.
assert ob["mean_pool_cosine"] > 0.999, "mean-pool must be order-blind"
assert ob["encoder_cosine"] < ob["mean_pool_cosine"], "encoder must beat mean-pool on order"

print(f'sentence A : "{se.ORDER_A}"')
print(f'sentence B : "{se.ORDER_B}"')
print(f"  mean-pool cosine : {ob['mean_pool_cosine']:.3f}   (order-blind -> IDENTICAL vector)")
print(f"  encoder   cosine : {ob['encoder_cosine']:.3f}   (order-aware -> below 1.000)")

sentence A : "The dog bit the man."
sentence B : "The man bit the dog."
  mean-pool cosine : 1.000   (order-blind -> IDENTICAL vector)
  encoder   cosine : 0.978   (order-aware -> below 1.000)


**Read it:** the mean-pool cosine is exactly `1.000` — the two sentences are *the same point in space*,
so any task that depends on word order (negation, who-did-what-to-whom) is invisible to it. That single
fact is why we need a model that *reads* the sentence, not just averages it.

## Step 2 — SIF: a principled average that down-weights frequent words

Before neural encoders, **SIF** (Arora, Liang & Ma 2017) asked: if we *must* average, can we average
**well**? Weight each word by $a/(a+p(w))$ — a smooth, learned-cutoff down-weighting of frequent words
— then subtract the corpus's common direction. On the toy sentence `"the cat sat"`, watch the stopword
`"the"` get pulled out of the sentence vector. (This is pure math — no model needed.)

In [3]:
ex = se.sif_worked_example()

print("word |  p(w)   | SIF weight a/(a+p)")
for w, p, raw in zip(ex["words"], ex["probs"], ex["weights"]):
    print(f"{w:>4} | {p:>7} | {raw:.4f}")

plain_x = ex["plain"][0]   # x-axis = the stopword 'the' direction (its vector is (1,0))
sif_x = ex["sif"][0]
# Contract: SIF reduces the stopword's pull on the sentence vector.
assert sif_x < plain_x, "SIF must reduce the stopword's pull"
print(f"\nplain-mean   x-coordinate (pull toward 'the') : {plain_x:.3f}")
print(f"SIF-weighted x-coordinate (pull toward 'the') : {sif_x:.3f}")
print(f"reduction in stopword pull                    : {plain_x / sif_x:.1f}x")

word |  p(w)   | SIF weight a/(a+p)
 the |    0.05 | 0.0196
 cat |  0.0003 | 0.7692
 sat |  0.0008 | 0.5556

plain-mean   x-coordinate (pull toward 'the') : 0.400
SIF-weighted x-coordinate (pull toward 'the') : 0.097
reduction in stopword pull                    : 4.1x


**Read it:** the plain mean sits at `x = 0.400` — the bland `"the"` vector drags it almost halfway
along the stopword axis even though `"the"` carries no topic. SIF cuts that to `x = 0.097`, a ~4×
reduction, and the vector now points at the **content** words (`cat`, `sat`). SIF fixes
*frequency-domination*; it still cannot fix *order-blindness* (it is still a weighted average) — for
that we need the encoder.

## Step 3 — the trained bi-encoder separates meaning

Now the headline. A **Sentence-BERT** bi-encoder is fine-tuned (siamese, shared weights) so that its
pooled vectors are **cosine-comparable**. Encode a paraphrase pair and an unrelated pair and compare:
the encoder pushes the paraphrase **high** and the unrelated pair **low**, while a mean-pool of static
vectors barely distinguishes them.

In [4]:
ps = se.paraphrase_separation(encoder)

print("paraphrase set:")
for i, s in enumerate(se.PARAPHRASE_SET):
    print(f"  s[{i}] = {s}")

enc_gap = ps["encoder_paraphrase"] - ps["encoder_unrelated"]
mp_gap = ps["meanpool_paraphrase"] - ps["meanpool_unrelated"]
# Contract: the trained encoder separates paraphrase from unrelated far better than mean-pool.
assert enc_gap > mp_gap, "encoder must separate paraphrase/unrelated better than mean-pool"

print(f"\n            paraphrase   unrelated   separation gap")
print(f"  encoder :   {ps['encoder_paraphrase']:+.3f}      {ps['encoder_unrelated']:+.3f}       {enc_gap:+.3f}")
print(f"  meanpool:   {ps['meanpool_paraphrase']:+.3f}      {ps['meanpool_unrelated']:+.3f}       {mp_gap:+.3f}")

paraphrase set:
  s[0] = A man is playing a guitar.
  s[1] = A person is strumming an acoustic guitar.
  s[2] = A child is feeding a baby elephant.

            paraphrase   unrelated   separation gap
  encoder :   +0.708      -0.089       +0.797
  meanpool:   +0.397      +0.624       -0.227


**Read it:** with a real backend the encoder gives paraphrase ≈ `0.71` and unrelated ≈ `−0.09` — a
clean, wide gap. The mean-pool baseline can even score the *unrelated* pair **higher** than the
paraphrase, because random static vectors only capture surface overlap, not meaning. *That separation
is the whole product* — search, clustering and dedup are just geometry on top of it.

## Step 4 — pooling strategy matters: mean vs max vs CLS

The encoder emits one vector **per token**; *pooling* collapses them to one sentence vector. The three
standard choices (mean / max / first-token "CLS") give **different** vectors and **different** scores.
We score each by **Spearman** rank correlation against gold similarities on a small STS-like set. Mean
pooling is the robust default — and usually wins.

In [5]:
pooling = se.pooling_comparison(encoder)

print("pooling -> STS Spearman (cosine vs gold):")
for strat in ("mean", "max", "cls"):
    print(f"  {strat:>4}: {pooling[strat]:+.3f}")

best = max(pooling, key=pooling.get)
print(f"\nbest on this set: {best!r}")

pooling -> STS Spearman (cosine vs gold):
  mean: +0.733
   max: +0.673
   cls: +0.685

best on this set: 'mean'


**Read it:** on a real backend, **mean** pooling tops the three — it incorporates every token, so no
single position is a bottleneck, and it is robust to length. `[CLS]` (the first-token vector) and max
trail. This is why nearly every production sentence encoder ships with **mean pooling**, and why you
must use the **same** pooling at index time and query time or your cosines are meaningless.

## Step 5 — semantic search: meaning, not keywords

A bi-encoder is the *retriever* behind modern search and RAG: embed the corpus **once**, then rank by
cosine to the query. The give-away that this is *meaning* matching, not keyword matching: the relevant
document that says `"forgot"` / `"Forgot password"` ranks high for the query `"reset … password"`
**without sharing the word "reset"**.

In [6]:
ranking = se.semantic_search(encoder)

# Contract: the top result is a password document (a meaning match, not a keyword match).
assert "password" in ranking[0][1].lower(), "top result should be a password doc"

print(f'query: "{se.SEARCH_QUERY}"\n')
for rank, (sim, doc) in enumerate(ranking, 1):
    print(f"  {rank}. {sim:+.3f}  {doc}")

query: "How do I reset my account password?"

  1. +0.711  To change your password, open Settings and click 'Reset password'.
  2. +0.618  If you forgot your login, use the 'Forgot password' link on the sign-in page.
  3. +0.080  Our refund policy allows returns within 30 days of purchase.
  4. +0.011  The mobile app supports dark mode and push notifications.
  5. -0.006  Premium plans include priority support and extra storage.


**Read it:** the two password documents sit far above everything else, and note the second one never
says `"reset"` — the embedding matched the *meaning* of password recovery. That is the win pure lexical
search (BM25) cannot buy. At scale you would unit-normalize the vectors and put them in an **ANN index**
(HNSW / IVF-PQ) so this cosine search runs in milliseconds over millions of documents.

## Step 5b — bi-encoder vs cross-encoder: same ranking, sharper scores

The **bi-encoder** above gives a reusable vector per text (cacheable, indexable → *retrieve*). A
**cross-encoder** instead feeds the *pair* jointly through one transformer for a single relevance score
(no reusable vector → *rerank*). On the same three documents, both rank the right one first, but the
cross-encoder's joint cross-attention spreads the scores **far** more sharply. (The cross-encoder is
loaded on demand; offline, the bi-encoder cosines still print and the cell degrades gracefully.)

In [7]:
bvc = se.bi_vs_cross_demo(encoder)

print(f'query: "{se.BI_VS_CROSS_QUERY}"\n')
print(f"{'bi cos':>8} | {'cross logit':>11} | document")
for i, (doc, bi) in enumerate(zip(bvc["docs"], bvc["bi_cosines"])):
    if bvc["cross_available"]:
        print(f"{bi:>+8.3f} | {bvc['cross_logits'][i]:>+11.2f} | {doc}")
    else:
        print(f"{bi:>+8.3f} | {'offline':>11} | {doc}")

if bvc["cross_available"]:
    # Both encoders rank the password doc first; the cross-encoder spreads its scores far more.
    assert bvc["bi_cosines"][0] == max(bvc["bi_cosines"])
    assert bvc["cross_logits"][0] == max(bvc["cross_logits"])
    bi_gap = bvc["bi_cosines"][0] - bvc["bi_cosines"][1]
    cross_gap = bvc["cross_logits"][0] - bvc["cross_logits"][1]
    print(f"\ntop-vs-2nd gap: bi-encoder {bi_gap:+.3f} (cosine)  vs  cross-encoder {cross_gap:+.2f} (logits) — far sharper")
else:
    print("\n(cross-encoder offline: bi-encoder cosines shown; the contrast holds when it loads)")

query: "How do I reset my account password?"

  bi cos | cross logit | document
  +0.711 |       +6.30 | To change your password, open Settings and click 'Reset password'.
  +0.642 |       -3.16 | If you forgot your login, use the 'Forgot password' link.
  +0.080 |      -11.10 | Our refund policy allows returns within 30 days of purchase.

top-vs-2nd gap: bi-encoder +0.068 (cosine)  vs  cross-encoder +9.46 (logits) — far sharper


**Read it:** the bi-encoder's top-vs-2nd gap is narrow (`0.711` vs `0.642`), but the cross-encoder's
joint attention catches that doc 2 is *login recovery* — a near-but-not-exact match — and confidently
spreads the logits (`+6.30` vs `−3.16` vs `−11.10`). That is why production search is **two stages**:
a bi-encoder retrieves the top ~100 from millions cheaply, then a cross-encoder **reranks** just those.
(Doc 2 here is a shortened string, so its bi cosine reads `0.642` vs `0.618` in Step 5's full-string
ranking — same document, slightly different text.)

## Step 6 — STS evaluation: does cosine track human similarity?

The intrinsic evaluation for sentence embeddings is **STS**: for each labelled pair compute the cosine,
then report the **Spearman rank correlation** with the human score. Spearman (rank-based) is the
convention because we care about *ordering* similarity correctly, not matching an absolute scale.

In [8]:
sts = se.sts_evaluation(encoder)

# Contract: cosine should rank-correlate positively with the gold similarity labels.
assert sts["spearman"] > 0.5, "cosine should track gold similarity"

print("gold | cosine")
for (a, b, g), c in zip(se.STS_PAIRS, sts["cosines"]):
    print(f"{g:>4.2f} | {c:+.3f}   ({a[:34]!r:36} <-> {b[:34]!r})")

print(f"\nSpearman(cosine, gold) = {sts['spearman']:.3f}")

gold | cosine
0.95 | +0.708   ('A man is playing a guitar.'         <-> 'A person is strumming an acoustic ')
0.85 | +0.414   ('A dog is running in the park.'      <-> 'A puppy sprints across the grass.')
0.80 | +0.687   ('The chef chopped fresh vegetables.' <-> 'A cook is slicing carrots and onio')
0.78 | +0.688   ('She booked a flight to Tokyo.'      <-> 'He reserved a plane ticket to Japa')
0.82 | +0.621   ('The stock market fell sharply toda' <-> 'Share prices dropped a lot this af')
0.05 | -0.089   ('A child is feeding a baby elephant' <-> 'A man is playing a guitar.')
0.04 | +0.026   ('The forecast predicts heavy rain.'  <-> 'The chef chopped fresh vegetables.')
0.08 | +0.019   ('She booked a flight to Tokyo.'      <-> 'The stock market fell sharply toda')
0.03 | +0.008   ('A dog is running in the park.'      <-> 'Policymakers hiked interest rates.')
0.88 | +0.633   ('The cat sat on the warm windowsill' <-> 'A feline rested by the sunny windo')

Spearman(cosine, gold) = 0.733


**Read it:** high-gold pairs get high cosine, low-gold pairs get low cosine, and the Spearman
correlation summarizes how well the *ranking* matches. On `all-MiniLM-L6-v2` this small set lands
around `0.73`. For a *retrieval* job you would instead report **Recall@k / nDCG** on real queries —
intrinsic STS and extrinsic retrieval can disagree, so always evaluate on the task you will run.

## Step 7 — the training objectives, as numbers

SBERT's vectors become comparable because a **proxy objective** shapes the space. Three of those
mechanisms are simple enough to evaluate by hand:

- **Triplet loss** $\max(0,\; d(a,p) - d(a,n) + \text{margin})$ — zero once the positive is closer than
  the negative *by the margin*; otherwise positive (pull $p$ in, push $n$ out).
- **InfoNCE** (SimCSE) — pull the positive together, push every in-batch negative apart.
- **ColBERT MaxSim** — per-token late interaction: each query token takes its max over doc tokens, summed.

In [9]:
# Triplet loss, the page's two worked cases (distance d = 1 - cosine, margin 0.3).
solved = se.triplet_loss(0.013, 0.884)     # positive already much closer -> zero loss
violated = se.triplet_loss(0.400, 0.200)   # negative is actually closer -> large loss
assert solved == 0.0 and abs(violated - 0.5) < 1e-9
print(f"triplet (solved)   d(a,p)=0.013 d(a,n)=0.884 -> L = {solved:.3f}")
print(f"triplet (violated) d(a,p)=0.400 d(a,n)=0.200 -> L = {violated:.3f}")

# InfoNCE for one anchor: a clear positive + three random negatives -> low loss.
rng = np.random.default_rng(0)
anchor = se._unit(rng.standard_normal(16))
positive = se._unit(anchor + 0.05 * rng.standard_normal(16))   # nearly aligned with the anchor
negatives = np.stack([se._unit(rng.standard_normal(16)) for _ in range(3)])
loss = se.info_nce_loss(anchor, positive, negatives, tau=0.05)
print(f"InfoNCE (aligned positive, 3 random negatives) -> L = {loss:.3f}  (small: the positive wins)")

# ColBERT late interaction: per-query-token MaxSim over doc tokens, then summed.
cb = se.colbert_maxsim()
print(f"\nColBERT MaxSim  query={cb['query_tokens']}  doc={cb['doc_tokens']}")
for i, qt in enumerate(cb["query_tokens"]):
    best = cb["doc_tokens"][int(cb["row_argmax"][i])]
    print(f"   {qt:>9} -> best doc token {best:>9}  (max {cb['row_max'][i]:+.2f})")
print(f"   MaxSim score = sum of row maxima = {cb['score']:.3f}")

triplet (solved)   d(a,p)=0.013 d(a,n)=0.884 -> L = 0.000
triplet (violated) d(a,p)=0.400 d(a,n)=0.200 -> L = 0.500
InfoNCE (aligned positive, 3 random negatives) -> L = 0.000  (small: the positive wins)

ColBERT MaxSim  query=['reset', 'my', 'password']  doc=['change', 'your', 'password', 'in', 'settings']
       reset -> best doc token    change  (max +0.86)
          my -> best doc token      your  (max +0.81)
    password -> best doc token  password  (max +1.00)
   MaxSim score = sum of row maxima = 2.669


**Read it:** the triplet loss is a **hinge** — exactly zero once the positive clears the margin, so the
model stops spending capacity on already-solved triplets. InfoNCE stays small when the positive is
well-aligned and the negatives are far. **ColBERT's MaxSim** scores a pair token-by-token: each query
token finds its single best-matching document token (the shared `password` token hits `1.00`), and
those maxima sum to the relevance score — fine-grained matching that stays indexable.

## Step 8 — the offline path always works

To prove the notebook never depends on the network, force the **synthetic** backend and re-run the two
load-bearing contracts. The synthetic encoder is not SBERT — but it is built to honour the same
qualitative shape (order matters; paraphrases close, unrelated far), so the teaching points survive
offline.

In [10]:
synth = se.load_encoder(force_synthetic=True)
print("forced backend:", synth.name, "(is_real =", synth.is_real, ")")

ob_s = se.order_blindness_demo(synth)
ps_s = se.paraphrase_separation(synth)
assert ob_s["mean_pool_cosine"] > 0.999            # still order-blind
assert ob_s["encoder_cosine"] < ob_s["mean_pool_cosine"]   # synthetic encoder is order-aware
assert ps_s["encoder_paraphrase"] > ps_s["encoder_unrelated"]  # paraphrase still beats unrelated
print(f"  order: mean-pool {ob_s['mean_pool_cosine']:.3f}  vs encoder {ob_s['encoder_cosine']:.3f}")
print(f"  paraphrase {ps_s['encoder_paraphrase']:.3f}  vs unrelated {ps_s['encoder_unrelated']:.3f}")
print("offline contracts held.")

forced backend: synthetic (is_real = False )
  order: mean-pool 1.000  vs encoder 0.939
  paraphrase 0.811  vs unrelated 0.201
offline contracts held.


## What we saw

- **Mean-pooling is order-blind** — `"dog bit man"` and `"man bit dog"` got the *identical* vector
  (cosine `1.000`); a trained encoder separates them. Averaging is a topic baseline, not a meaning one.
- **SIF** down-weights frequent words ($a/(a+p(w))$) and cut the stopword's pull ~4× on `"the cat sat"`.
- **A trained bi-encoder** pushes paraphrases high and unrelated low by cosine — the separation that
  powers search, clustering and dedup.
- **Mean pooling** topped max and CLS on STS Spearman — the robust default.
- **Semantic search** ranked the right document first *without keyword overlap*; the **cross-encoder**
  reranked the same docs with far sharper scores; **STS Spearman** ≈ 0.73 shows cosine tracking meaning.
- The **triplet** loss is a hinge, **InfoNCE** rewards aligned positives and far negatives, and
  **ColBERT MaxSim** scores token-by-token — the objectives that make the space comparable.

Every number here is reproduced by `sentence_embeddings.py` and drawn by `make_figures_07.py`, so the
page, the figures, and this notebook are one consistent story. The full explanation — SBERT's siamese
training, the anisotropy fix, SimCSE, ColBERT, MTEB — lives in the concept page next to this notebook.